In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")

In [ ]:
# === Cell 2: Imports ===

%matplotlib inline

import json
import sys
import platform
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import spsolve

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import joblib
import sklearn

from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# === Cell 3: Config ===

PROJECT_NAME = "20260826_Classifier_Generalizability_Assessment_VLIZ"

# --- Paths ---
DATASET_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/VLIZ_Plastic_Reflectance")
XLSX_PATH    = DATASET_ROOT / "Leone_et_al__2021_hyperspectral_data_updated_version2.xlsx"

PROJECT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability") / PROJECT_NAME
RESULTS_DIR  = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Model and reference axis (set before running) ---
# C2 variant (cut + ALS + z-score), the preprocessing chain this notebook reproduces.
# REFERENCE_AXIS_CSV: any preprocessed training spectrum of the same variant; only its
# wavenumber column is read to recover the training grid used by the axis guard.
BENCHMARK_ROOT = Path("")   # root of the preprocessing-influence benchmark

# C2 variant
MODEL_PATH         = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_LR_Deployment_Training/results/models/20260629_LR_Deployment_Training_LR_deployment.joblib")
REFERENCE_AXIS_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score/HDPE_obj_01_body")

CLASSIFIER_NAME = "LR"

# Optional; enables the training overlays in the spectral overlay and PCA plot cells.
TRAINING_SPECTRA_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score") 
IR_GROUND_TRUTH_CSV  = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_NIR_Dataset/spectra_ground_truth_v8.csv")

# --- Preprocessing (same defaults as the IR preprocessing notebook) ---
# KI-Sort and VLIZ spectra are reflectance, so the ALS peak direction is "negative".
DEFAULT_ALS = {"lam": 1_000_000.0, "p": 0.01, "niter": 10}
ALS_PEAK_DIRECTION    = "negative"
ALS_ZERO_ANCHOR_Q     = 0.995
ALS_ONLY_SHIFT_DOWN   = True

# --- Dataset semantics ---
ALL_CLASSES    = ["PE", "PET", "PP", "PS"]
MATERIAL_ORDER = ["PE", "PET", "PP", "PS"]

# microFTIR string -> KI-Sort class. Keys are lowercased+stripped before lookup;
# the source column carries both "Polyethylene terephthalate (PET)" and a
# lowercase-p variant of the same string.
FTIR_TO_CLASS = {
    "polyethylene (pe)":                 "PE",
    "polyethylene terephthalate (pet)":  "PET",
    "polypropylene (pp)":                "PP",
    "polystyrene (ps)":                  "PS",
}
# FTIR-confirmed polymers without a KI-Sort class; assigned to the `ood` tier.
FTIR_OUT_OF_SCOPE = ["fluorocarbon", "thermoplastic elastomer", "paraffin"]
# Non-identifications; these rows are counted and excluded.
FTIR_UNRESOLVED = ["nd", "na"]

FIELD_ORIGINS    = ["Port of Antwerp (PoA)", "Temse Bridge (TB)"]
PRISTINE_ORIGINS = ["Carat"]

# Only dry measurements are used (no wet, partially wet or submerged states).
KEEP_STATES = ["dry"]
# True additionally admits Weathered / Biofouled specimens into the pristine tier.
INCLUDE_TREATED = False

# Classes with fewer distinct items get no pooled accuracy in the summary CSV,
# only per-item results (same value as MIN_OBJECTS_PER_GROUP in the metadata analysis).
MIN_ITEMS_FOR_RATE = 3

# --- ASD splice diagnostic ---
# The FieldSpec 4 joins three detectors at nominal boundaries ~1000 nm (VNIR/SWIR1)
# and ~1830 nm (SWIR1/SWIR2); the distributed data are not splice-corrected.
SPLICE_WAVELENGTHS_NM = [1000.0, 1830.0]
SPLICE_WINDOW_NM      = 5.0

# Splice correction (method chosen in this project; the dataset authors prescribe none):
# additive offset per boundary so the upper segment's edge level matches the lower one,
# VNIR held fixed, offsets accumulated left to right; band shapes are preserved.
SPLICE_CORRECTION_ENABLED = True
# Points on each side of a boundary whose median estimates the step.
SPLICE_ANCHOR_POINTS = 5
# A jump is flagged above this multiple of the spectrum's median abs. first difference.
SPLICE_JUMP_FACTOR    = 8.0

RANDOM_SEED = 42

RUN_TS = f"{datetime.now():%Y%m%d_%H%M%S}"

print(f"PROJECT_NAME: {PROJECT_NAME}")
print(f"RESULTS_DIR:  {RESULTS_DIR}")
print(f"RUN_TS:       {RUN_TS}")
print(f"python {platform.python_version()} | sklearn {sklearn.__version__} | numpy {np.__version__}")
for _label, _p in [("XLSX_PATH", XLSX_PATH),
                   ("MODEL_PATH", MODEL_PATH),
                   ("REFERENCE_AXIS_CSV", REFERENCE_AXIS_CSV),
                   ("TRAINING_SPECTRA_DIR", TRAINING_SPECTRA_DIR)]:
    if str(_p) == "" or str(_p) == ".":
        print(f"  UNPATCHED: {_label}")
    elif not _p.exists():
        print(f"  MISSING:   {_label} -> {_p}")
    else:
        print(f"  ok:        {_label}")


In [ ]:
# === Cell 4: Shared Functions and Plot Style ===

# Preprocessing functions are identical to the IR preprocessing notebook; keep them in
# sync so the ALS baseline matches the training data.

def cut_spectrum(df, lower, upper):
    lo = min(lower, upper)
    hi = max(lower, upper)
    mask = (df["wavenumber"] >= lo) & (df["wavenumber"] <= hi)
    cut_df = df.loc[mask, ["wavenumber", "signal"]].copy().reset_index(drop=True)
    if cut_df.empty:
        raise ValueError(f"Empty cut range for bounds {lo} to {hi} 1/cm.")
    return cut_df


def als_baseline(y, lam=1e7, p=0.01, niter=10, peak_direction="negative", tol=0):
    y = np.asarray(y, dtype=float)

    if y.ndim != 1:
        raise ValueError("Input spectrum must be one-dimensional.")
    if lam <= 0:
        raise ValueError("lam must be > 0.")
    if not (0 < p < 1):
        raise ValueError("p must be between 0 and 1.")
    if niter < 1:
        raise ValueError("niter must be >= 1.")
    if peak_direction not in {"positive", "negative"}:
        raise ValueError("peak_direction must be 'positive' or 'negative'.")

    m = y.size
    if m < 3:
        raise ValueError("Spectrum must contain at least 3 points for ALS baseline correction.")

    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(m - 2, m), format="csc")
    w = np.ones(m, dtype=float)

    for _ in range(int(niter)):
        W = sparse.spdiags(w, 0, m, m)
        Z = W + lam * (D.T @ D)
        z = spsolve(Z, w * y)

        if peak_direction == "positive":
            w_new = p * (y > z) + (1 - p) * (y <= z)
        else:  # negative peaks
            w_new = p * (y < z) + (1 - p) * (y >= z)

        if np.allclose(w, w_new, atol=tol, rtol=0):
            w = w_new
            break
        w = w_new

    return z, w


def baseline_correct_spectrum(
    df,
    lam,
    p,
    niter,
    peak_direction="negative",
    zero_anchor_quantile=0.995,
    only_shift_down=True,
):
    baseline, weights = als_baseline(
        df["signal"].to_numpy(),
        lam=lam,
        p=p,
        niter=niter,
        peak_direction=peak_direction,
    )

    corrected = df.copy()
    corrected["signal"] = corrected["signal"].to_numpy() - baseline

    anchor_value = float(np.quantile(corrected["signal"], zero_anchor_quantile))

    if only_shift_down:
        anchor_shift = max(anchor_value, 0.0)
    else:
        anchor_shift = anchor_value

    corrected["signal"] = corrected["signal"] - anchor_shift

    baseline_df = df[["wavenumber"]].copy()
    baseline_df["signal"] = baseline

    diagnostics = {
        "anchor_shift": anchor_shift,
        "q50": float(np.quantile(corrected["signal"], 0.50)),
        "q95": float(np.quantile(corrected["signal"], 0.95)),
        "q99": float(np.quantile(corrected["signal"], 0.99)),
        "fraction_above_zero": float(np.mean(corrected["signal"].to_numpy() > 0)),
        "peak_direction": peak_direction,
    }

    return corrected, baseline_df, weights, diagnostics


def zscore_normalize_spectrum(df):
    values = df["signal"].to_numpy(dtype=float)
    mean = float(values.mean())
    std = float(values.std(ddof=0))
    notes = []
    if np.isclose(std, 0.0):
        normalized_values = np.zeros_like(values)
        notes.append("zero_std_detected")
    else:
        normalized_values = (values - mean) / std
    normalized = df.copy()
    normalized["signal"] = normalized_values
    return normalized, {"mean": mean, "std": std, "notes": notes}


def correct_splice(nm_asc, signal_asc, boundaries_nm, n_anchor=5):
    """Additive-offset ASD splice correction, applied on the nm-ascending axis.

    For each boundary the upper segment is shifted by a constant so its edge
    level matches the lower segment's edge level; offsets accumulate left to
    right with the VNIR segment held fixed. The step is estimated as the
    difference of medians over n_anchor points either side of the boundary.

    Returns (corrected_signal, offsets) where offsets maps boundary -> applied
    shift, so the magnitude of every correction stays auditable.
    """
    nm = np.asarray(nm_asc, dtype=float)
    y = np.asarray(signal_asc, dtype=float).copy()
    offsets = {}
    for b in sorted(float(x) for x in boundaries_nm):
        # Last index belonging to the lower segment.
        lower_idx = np.searchsorted(nm, b, side="right") - 1
        if lower_idx < 0 or lower_idx >= nm.size - 1:
            offsets[b] = 0.0
            continue
        lo0 = max(0, lower_idx - n_anchor + 1)
        hi1 = min(nm.size, lower_idx + 1 + n_anchor)
        lower_level = float(np.median(y[lo0:lower_idx + 1]))
        upper_level = float(np.median(y[lower_idx + 1:hi1]))
        shift = lower_level - upper_level
        y[lower_idx + 1:] += shift
        offsets[b] = shift
    return y, offsets


def nm_to_wavenumber(nm):
    """Convert wavelength in nm to wavenumber in 1/cm."""
    return 1.0e7 / np.asarray(nm, dtype=float)


def resample_to_axis(src_wn, src_signal, ref_wn):
    """Linear-interpolate a spectrum onto ref_wn. Both axes must be ascending.

    np.interp extrapolates as a flat plateau outside the source range; callers
    range-check first (Reference Axis and Range Check cell).
    """
    return np.interp(np.asarray(ref_wn, dtype=float),
                     np.asarray(src_wn, dtype=float),
                     np.asarray(src_signal, dtype=float))


def preprocess_spectrum(wn_asc, signal_asc, ref_wn):
    """cut-by-resample -> ALS baseline -> z-score. Returns the feature vector."""
    resampled = resample_to_axis(wn_asc, signal_asc, ref_wn)
    df = pd.DataFrame({"wavenumber": ref_wn, "signal": resampled})
    corrected, _, _, _ = baseline_correct_spectrum(
        df,
        lam=DEFAULT_ALS["lam"],
        p=DEFAULT_ALS["p"],
        niter=DEFAULT_ALS["niter"],
        peak_direction=ALS_PEAK_DIRECTION,
        zero_anchor_quantile=ALS_ZERO_ANCHOR_Q,
        only_shift_down=ALS_ONLY_SHIFT_DOWN,
    )
    normalized, _ = zscore_normalize_spectrum(corrected)
    return normalized["signal"].to_numpy(dtype=float)


def _patch_missing_multi_class(estimator):
    """Add the `multi_class` attribute to a LogisticRegression pickled with sklearn 1.8.0.

    sklearn 1.7.2's predict_proba() requires it. No-op for other estimators."""
    step = estimator.steps[-1][1] if hasattr(estimator, "steps") else estimator
    if type(step).__name__ == "LogisticRegression" and not hasattr(step, "multi_class"):
        step.multi_class = "auto"
        print(f"  Patched missing 'multi_class' on {type(step).__name__} "
              f"(sklearn {sklearn.__version__} runtime vs. newer pickling version).")


# --- Plot style and shared plotting helpers ---
# Defined here so each plotting cell runs standalone after the setup cells.
_FW = 16 / 2.54
_FONT_SIZE = 10
_BAND_COLOR = "#f0f0f0"

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# Material colours: tab10 in MATERIAL_ORDER.
MATERIAL_COLORS = {
    "PE":    "#1f77b4",
    "PET":   "#ff7f0e",
    "PP":    "#2ca02c",
    "PS":    "#d62728",
    "Paper": "#9467bd",   # deployment model has 5 classes, not the 4 in MATERIAL_ORDER
}
_OTHER_COLOR = "#7f7f7f"

# Classes the classifier can predict. MATERIAL_ORDER is the ground-truth order; VLIZ
# has no Paper items, but "Paper" can still be predicted and is kept in plots.
PRED_CLASS_ORDER = MATERIAL_ORDER + ["Paper"]


def _add_bands(ax, n):
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_BAND_COLOR, zorder=0)


_TIER_TITLE = {"field": "Field", "pristine": "Pristine",
               "ood": "Out Of Distribution"}


def _short(s, n=20):
    """Truncate a long item key for use as a tick label."""
    s = str(s)
    return s if len(s) <= n else s[: n - 1] + "…"


def _latest(pattern):
    """Resolve a results file by exact name, falling back to the newest glob match."""
    exact = RESULTS_DIR / pattern.replace("_*", "")
    if exact.exists():
        return exact
    hits = sorted(RESULTS_DIR.glob(pattern))
    return hits[-1] if hits else None


print("preprocessing chain: raw reflectance -> splice correction -> resample to "
      "training axis -> ALS baseline -> SNV")
print(f"splice correction enabled: {SPLICE_CORRECTION_ENABLED}, "
      f"boundaries {SPLICE_WAVELENGTHS_NM} nm, anchor points {SPLICE_ANCHOR_POINTS}")
print(f"ALS: {DEFAULT_ALS}, peak_direction={ALS_PEAK_DIRECTION}")


def _load_training_matrix():
    """Return (X, y) for the training spectra folder, class per file resolved
    from the GT CSV by object stem + region, with the filename material prefix
    as fallback. Files that resolve to neither are skipped and counted.

    Training spectra files have no extension (one wavenumber;signal pair per
    line, no header), so files are selected with is_file().
    """
    if str(TRAINING_SPECTRA_DIR) in ("", "."):
        return None, None, 0
    files = sorted(p for p in Path(TRAINING_SPECTRA_DIR).iterdir() if p.is_file())
    if not files:
        return None, None, 0
    gt = None
    if IR_GROUND_TRUTH_CSV.exists():
        gt = pd.read_csv(IR_GROUND_TRUTH_CSV, sep=";")
        _fn = "filename" if "filename" in gt.columns else gt.columns[0]
        gt = gt.set_index(gt[_fn].astype(str).str.strip())
    region_col = {"body": "body_class", "cap": "cap_class", "label": "label_class"}
    prefix_alias = {"HDPE": "PE", "LDPE": "PE", "PAPER": "Paper"}
    X, y, skipped = [], [], 0
    for f in files:
        stem = f.stem if f.suffix else f.name
        cls = None
        parts = stem.split("_")
        region = next((p for p in parts if p in region_col), None)
        if gt is not None and region is not None:
            obj = stem.split(f"_{region}")[0]
            if obj in gt.index:
                v = gt.loc[obj, region_col[region]]
                v = v.iloc[0] if hasattr(v, "iloc") else v
                if pd.notna(v) and str(v).strip():
                    cls = str(v).strip()
        if cls is None:
            pref = parts[0].upper()
            cls = prefix_alias.get(pref, pref)
        if cls not in PRED_CLASS_ORDER:
            skipped += 1
            continue
        d = pd.read_csv(f, sep=";", header=None,
                        names=["wavenumber", "reflectance"], engine="python")
        d["wavenumber"] = pd.to_numeric(d["wavenumber"], errors="coerce")
        d["reflectance"] = pd.to_numeric(d["reflectance"], errors="coerce")
        d = d.dropna().sort_values("wavenumber")
        X.append(d["reflectance"].to_numpy(dtype=float))
        y.append(cls)
    if not X:
        return None, None, skipped
    return np.vstack(X), np.array(y), skipped


In [ ]:
# === Cell 5: Load Workbook ===

# `data` sheet layout: column A empty, column B the wavelength axis in nm, and
# one column per spectrum headed by an integer ID. `Metadata` carries one row
# per spectrum ID (pseudo-replicates each get their own row), joined on
# `ID-data sheet`.

_raw_data = pd.read_excel(XLSX_PATH, sheet_name="data", header=0)
_raw_meta = pd.read_excel(XLSX_PATH, sheet_name="Metadata ", header=0)

# Wavelength axis: second column of the data sheet.
WAVELENGTH_NM = _raw_data.iloc[:, 1].to_numpy(dtype=float)

# Spectrum columns: everything from the third column on. Headers arrive as ints
# or int-like floats depending on the reader; normalise to int.
_spec_block = _raw_data.iloc[:, 2:]
_spec_ids = []
for c in _spec_block.columns:
    try:
        _spec_ids.append(int(float(c)))
    except (TypeError, ValueError):
        _spec_ids.append(None)
_keep = [i for i, v in enumerate(_spec_ids) if v is not None]
SPECTRA = _spec_block.iloc[:, _keep].to_numpy(dtype=float).T          # (n_spectra, n_wavelengths)
SPECTRUM_IDS = np.array([_spec_ids[i] for i in _keep], dtype=int)

# Metadata: drop the trailing all-empty columns, strip the trailing spaces the
# source workbook carries in several header names.
_raw_meta = _raw_meta.loc[:, [c for c in _raw_meta.columns if not str(c).startswith("Unnamed")]]
_raw_meta.columns = [str(c).strip() for c in _raw_meta.columns]
META = _raw_meta[_raw_meta["Sample code"].notna()].copy()
META["ID-data sheet"] = META["ID-data sheet"].astype(int)

# Normalised working columns.
META["ftir_norm"]  = META["Polymer identified via microFTIR"].astype(str).str.strip().str.lower()
META["state_norm"] = META["State"].astype(str).str.strip().str.lower()
META["type_norm"]  = META["Type"].astype(str).str.strip().str.lower()
META["origin"]     = META["Origin"].astype(str).str.strip()

_id_to_row = {int(v): i for i, v in enumerate(SPECTRUM_IDS)}

print(f"data sheet:     {SPECTRA.shape[0]} spectra x {SPECTRA.shape[1]} wavelengths")
print(f"wavelength:     {WAVELENGTH_NM.min():.0f}-{WAVELENGTH_NM.max():.0f} nm "
      f"({nm_to_wavenumber(WAVELENGTH_NM.max()):.0f}-{nm_to_wavenumber(WAVELENGTH_NM.min()):.0f} 1/cm)")
print(f"metadata:       {len(META)} rows, {META['ID-data sheet'].nunique()} unique IDs")
print(f"ids joinable:   {META['ID-data sheet'].isin(_id_to_row.keys()).sum()} / {len(META)}")
print()
print("State:"); print(META["state_norm"].value_counts().to_string())
print()
print("Type:");  print(META["type_norm"].value_counts().to_string())


In [ ]:
# === Cell 6: Tier Assignment and Ground Truth ===

# Ground truth is the microFTIR column. `Polymer commercial labeling` is kept for
# traceability; label_agrees_ftir marks where it matches the FTIR class.

_sel = META[META["state_norm"].isin([s.lower() for s in KEEP_STATES])].copy()

_n_unresolved = _sel["ftir_norm"].isin(FTIR_UNRESOLVED).sum()
_sel = _sel[~_sel["ftir_norm"].isin(FTIR_UNRESOLVED)].copy()

_sel["true_class"]  = _sel["ftir_norm"].map(FTIR_TO_CLASS)
_sel["in_scope"]    = _sel["true_class"].notna()
_sel["ftir_polymer"] = _sel["Polymer identified via microFTIR"].astype(str).str.strip()

_treated = ["weathered", "biofouled", "biofilm"]


def _assign_tier(row):
    if row["ftir_norm"] in FTIR_OUT_OF_SCOPE:
        return "ood"
    if not row["in_scope"]:
        return None
    is_field = ("field" in row["type_norm"]) or (row["origin"] in FIELD_ORIGINS)
    if is_field:
        return "field"
    is_treated = any(t in row["type_norm"] for t in _treated)
    if is_treated and not INCLUDE_TREATED:
        return None
    if row["origin"] in PRISTINE_ORIGINS or "pristine" in row["type_norm"]:
        return "pristine"
    return None


_sel["tier"] = _sel.apply(_assign_tier, axis=1)
_n_untiered = _sel["tier"].isna().sum()
SEL = _sel[_sel["tier"].notna()].copy()

# Item key: one physical specimen.
#   field:    leading token of the sample code + origin (an item appears under several
#             sample codes, one per tank session)
#   pristine: commercial label only; `Replicates` cannot reliably separate physical
#             sheets from sessions, so each reference polymer is one item (below
#             MIN_ITEMS_FOR_RATE, so no pooled rates)
#   ood:      leading token + origin, as for field
_lead = SEL["Sample code"].astype(str).str.split("_").str[0]
SEL["item_key"] = np.where(
    SEL["tier"] == "pristine",
    SEL["Polymer commercial labeling"].astype(str).str.strip(),
    _lead + "_" + SEL["origin"],
)

SEL["commercial_label"] = SEL["Polymer commercial labeling"].astype(str).str.strip()
SEL["ftir_score"] = pd.to_numeric(
    SEL["microFTIR polymer library matching score (%)"], errors="coerce")
SEL["row_idx"] = SEL["ID-data sheet"].map(_id_to_row)
_n_no_spectrum = SEL["row_idx"].isna().sum()
SEL = SEL[SEL["row_idx"].notna()].copy()
SEL["row_idx"] = SEL["row_idx"].astype(int)

ITEMS = (SEL.groupby(["tier", "item_key"])
            .agg(true_class=("true_class", "first"),
                 ftir_polymer=("ftir_polymer", "first"),
                 commercial_label=("commercial_label", "first"),
                 origin=("origin", "first"),
                 sample_type=("Type", "first"),
                 ftir_score=("ftir_score", "max"),
                 n_spectra=("ID-data sheet", "size"),
                 n_sample_codes=("Sample code", "nunique"))
            .reset_index())
ITEMS["label_agrees_ftir"] = [
    (str(c).upper().startswith(str(t)) if isinstance(t, str) else False)
    for c, t in zip(ITEMS["commercial_label"], ITEMS["true_class"])
]

print(f"excluded, FTIR unresolved (ND/NA): {_n_unresolved} rows")
print(f"excluded, no tier match:           {_n_untiered} rows")
print(f"excluded, no spectrum column:      {_n_no_spectrum} rows")
print(f"selected: {len(SEL)} spectra -> {len(ITEMS)} items")
print()
for _t in ["field", "pristine", "ood"]:
    _sub = ITEMS[ITEMS["tier"] == _t]
    if _sub.empty:
        print(f"[{_t}] none")
        continue
    _by = _sub.groupby(_sub["true_class"].fillna(_sub["ftir_polymer"])).size()
    print(f"[{_t}] {len(_sub)} items: " + ", ".join(f"{k}={v}" for k, v in _by.items()))
display(ITEMS)


In [ ]:
# === Cell 7: Reference Axis and Range Check ===

# Training grid read from a preprocessed training spectrum; the feature-length guard
# in the Classify cell checks against it. The splice diagnostic uses this window.

if str(REFERENCE_AXIS_CSV) in ("", "."):
    raise ValueError("REFERENCE_AXIS_CSV is unpatched -- see Cell 3.")

_ref = pd.read_csv(REFERENCE_AXIS_CSV, sep=";", header=None,
                   names=["wavenumber", "reflectance"], engine="python")
_ref["wavenumber"] = pd.to_numeric(_ref["wavenumber"], errors="coerce")
_ref = _ref.dropna(subset=["wavenumber"]).sort_values("wavenumber").reset_index(drop=True)
REF_WN = _ref["wavenumber"].to_numpy(dtype=float)

# Training window expressed in nm, for the splice check below.
REF_NM_MIN = float(1.0e7 / REF_WN.max())
REF_NM_MAX = float(1.0e7 / REF_WN.min())

# VLIZ native axis, ascending in wavenumber (nm ascending -> 1/cm descending).
SRC_WN_ASC = nm_to_wavenumber(WAVELENGTH_NM)
SRC_ORDER = np.argsort(SRC_WN_ASC)
SRC_WN_ASC = SRC_WN_ASC[SRC_ORDER]

_below = int((REF_WN < SRC_WN_ASC.min()).sum())
_above = int((REF_WN > SRC_WN_ASC.max()).sum())

print(f"training axis:  {REF_WN.min():.1f}-{REF_WN.max():.1f} 1/cm, {REF_WN.size} points")
print(f"                = {REF_NM_MIN:.1f}-{REF_NM_MAX:.1f} nm")
print(f"VLIZ axis:      {SRC_WN_ASC.min():.1f}-{SRC_WN_ASC.max():.1f} 1/cm, {SRC_WN_ASC.size} points")
if _below or _above:
    print(f"  WARNING: {_below} training points below and {_above} above the VLIZ range.")
    print("  np.interp clamps flat outside the source range -- those features would be")
    print("  a fabricated plateau, not data. Narrow the reference axis before citing results.")
else:
    print("  training window fully inside the VLIZ range; no extrapolation.")


In [ ]:
# === Cell 8: ASD Splice Diagnostic ===

# Flags detector steps at the nominal splice wavelengths (jump / median abs. first
# difference > SPLICE_JUMP_FACTOR), before and after correction, and whether each
# boundary lies inside the training window. Steps outside the window are cut away.

_rows = []
for _, r in SEL.iterrows():
    y = SPECTRA[int(r["row_idx"])]
    d = np.diff(y)
    scale = float(np.median(np.abs(d)))
    y_corr = (correct_splice(WAVELENGTH_NM, y, SPLICE_WAVELENGTHS_NM,
                             n_anchor=SPLICE_ANCHOR_POINTS)[0]
              if SPLICE_CORRECTION_ENABLED else y)
    d_corr = np.diff(y_corr)
    scale_corr = float(np.median(np.abs(d_corr)))
    rec = {
        "id": int(r["ID-data sheet"]),
        "item_key": r["item_key"],
        "tier": r["tier"],
        "median_abs_diff": scale,
    }
    flagged_any, flagged_in_window = False, False
    for wl in SPLICE_WAVELENGTHS_NM:
        m = (WAVELENGTH_NM[:-1] >= wl - SPLICE_WINDOW_NM) & (WAVELENGTH_NM[:-1] <= wl + SPLICE_WINDOW_NM)
        jump = float(np.max(np.abs(d[m]))) if m.any() else np.nan
        ratio = jump / scale if (scale > 0 and np.isfinite(jump)) else np.nan
        in_window = bool(REF_NM_MIN <= wl <= REF_NM_MAX)
        jump_c = float(np.max(np.abs(d_corr[m]))) if m.any() else np.nan
        ratio_c = (jump_c / scale_corr) if (scale_corr > 0 and np.isfinite(jump_c)) else np.nan
        rec[f"max_jump_{int(wl)}nm"] = jump
        rec[f"jump_ratio_{int(wl)}nm"] = ratio
        rec[f"jump_ratio_{int(wl)}nm_corrected"] = ratio_c
        rec[f"in_training_window_{int(wl)}nm"] = in_window
        if np.isfinite(ratio) and ratio > SPLICE_JUMP_FACTOR:
            flagged_any = True
            if in_window:
                flagged_in_window = True
    rec["jump_flagged"] = flagged_any
    rec["jump_flagged_in_window"] = flagged_in_window
    _rows.append(rec)

SPLICE_DF = pd.DataFrame(_rows)
_splice_csv = RESULTS_DIR / f"{PROJECT_NAME}_splice_diagnostic.csv"
SPLICE_DF.to_csv(_splice_csv, index=False)

print(f"spectra checked: {len(SPLICE_DF)}")
print(f"training window: {REF_NM_MIN:.1f}-{REF_NM_MAX:.1f} nm")
for wl in SPLICE_WAVELENGTHS_NM:
    _c = f"jump_ratio_{int(wl)}nm"
    _in = bool(REF_NM_MIN <= wl <= REF_NM_MAX)
    _n = int((SPLICE_DF[_c] > SPLICE_JUMP_FACTOR).sum())
    _cc = f"jump_ratio_{int(wl)}nm_corrected"
    _nc = int((SPLICE_DF[_cc] > SPLICE_JUMP_FACTOR).sum())
    print(f"  {int(wl)} nm ({'INSIDE' if _in else 'outside'} window): "
          f"jump/noise median={SPLICE_DF[_c].median():.2f} max={SPLICE_DF[_c].max():.2f}, "
          f"flagged {_n}/{len(SPLICE_DF)}")
    print(f"           after correction: median={SPLICE_DF[_cc].median():.2f} "
          f"max={SPLICE_DF[_cc].max():.2f}, flagged {_nc}/{len(SPLICE_DF)}")

SPLICE_DF["jump_flagged_in_window_corrected"] = False
for wl in SPLICE_WAVELENGTHS_NM:
    if REF_NM_MIN <= wl <= REF_NM_MAX:
        SPLICE_DF["jump_flagged_in_window_corrected"] |= (
            SPLICE_DF[f"jump_ratio_{int(wl)}nm_corrected"] > SPLICE_JUMP_FACTOR)
SPLICE_DF.to_csv(_splice_csv, index=False)

_n_in = int(SPLICE_DF["jump_flagged_in_window"].sum())
_n_in_c = int(SPLICE_DF["jump_flagged_in_window_corrected"].sum())
print(f"in-window flags: {_n_in} raw -> {_n_in_c} after correction")
if _n_in == 0:
    print("-> no flagged splice step inside the training window; the cut removes them.")
else:
    print(f"-> {_n_in} spectra carry a flagged splice step INSIDE the training window.")
print(f"saved: {_splice_csv.name}")


In [ ]:
# === Cell 9: Preprocess Spectra ===
#   X_SPEC : each pseudo-replicate preprocessed separately (stability output only)
#   X_ITEM : raw reflectance averaged per item, then preprocessed once (as recommended
#            by the dataset authors); used for the item-level predictions
# Splice correction is applied per spectrum before reordering, resampling or averaging.

SPLICE_OFFSETS = []


def _prep_raw(row_idx):
    """Fetch one raw spectrum on the nm-ascending axis, splice-corrected."""
    y_nm = SPECTRA[int(row_idx)]
    if SPLICE_CORRECTION_ENABLED:
        y_nm, offs = correct_splice(WAVELENGTH_NM, y_nm, SPLICE_WAVELENGTHS_NM,
                                    n_anchor=SPLICE_ANCHOR_POINTS)
        SPLICE_OFFSETS.append({"row_idx": int(row_idx),
                               **{f"offset_{int(k)}nm": v for k, v in offs.items()}})
    return y_nm


_spec_rows, _spec_meta = [], []
for _, r in SEL.iterrows():
    y = _prep_raw(r["row_idx"])[SRC_ORDER]
    _spec_rows.append(preprocess_spectrum(SRC_WN_ASC, y, REF_WN))
    _spec_meta.append({
        "id": int(r["ID-data sheet"]),
        "item_key": r["item_key"],
        "tier": r["tier"],
        "true_class": r["true_class"],
        "ftir_polymer": r["ftir_polymer"],
    })
X_SPEC = np.vstack(_spec_rows)
SPEC_META = pd.DataFrame(_spec_meta)

_item_rows = []
for _, it in ITEMS.iterrows():
    m = (SEL["tier"] == it["tier"]) & (SEL["item_key"] == it["item_key"])
    raw = np.vstack([_prep_raw(ri) for ri in SEL.loc[m, "row_idx"].to_numpy(dtype=int)])
    _item_rows.append(preprocess_spectrum(SRC_WN_ASC, raw.mean(axis=0)[SRC_ORDER], REF_WN))
X_ITEM = np.vstack(_item_rows)

_proc = pd.DataFrame(X_ITEM, columns=[f"{w:.2f}" for w in REF_WN])
_proc.insert(0, "true_class", ITEMS["true_class"].to_numpy())
_proc.insert(0, "item_key", ITEMS["item_key"].to_numpy())
_proc.insert(0, "tier", ITEMS["tier"].to_numpy())
_proc_csv = RESULTS_DIR / f"{PROJECT_NAME}_processed_item_spectra.csv"
_proc.to_csv(_proc_csv, index=False)

if SPLICE_CORRECTION_ENABLED and SPLICE_OFFSETS:
    _off = pd.DataFrame(SPLICE_OFFSETS).drop_duplicates(subset="row_idx")
    _off_csv = RESULTS_DIR / f"{PROJECT_NAME}_splice_offsets.csv"
    _off.to_csv(_off_csv, index=False)
    for c in [c for c in _off.columns if c.startswith("offset_")]:
        print(f"splice {c}: median={_off[c].median():+.5f} "
              f"min={_off[c].min():+.5f} max={_off[c].max():+.5f}")
    print(f"saved:  {_off_csv.name}")

print(f"X_SPEC: {X_SPEC.shape}  (per pseudo-replicate)")
print(f"X_ITEM: {X_ITEM.shape}  (per item, raw-averaged first)")
print(f"saved:  {_proc_csv.name}")


In [ ]:
# === Cell 10: Classify and Export Metrics ===

if str(MODEL_PATH) in ("", "."):
    raise ValueError("MODEL_PATH is unpatched -- see Cell 3.")

model = joblib.load(MODEL_PATH)
_patch_missing_multi_class(model)
MODEL_CLASSES = [str(c) for c in model.classes_]
print(f"model: {MODEL_PATH.name}")
print(f"classes: {MODEL_CLASSES}")

# Guard 1: feature length must match the training axis.
_n_expected = getattr(model, "n_features_in_", X_ITEM.shape[1])
if X_ITEM.shape[1] != _n_expected:
    raise ValueError(
        f"Feature length {X_ITEM.shape[1]} != model n_features_in_ {_n_expected}. "
        "REFERENCE_AXIS_CSV does not match the axis this model was trained on.")

# Guard 2: warn about GT classes the model cannot predict (expected for the ood tier).
_unknown = sorted(set(ITEMS["true_class"].dropna()) - set(MODEL_CLASSES))
if _unknown:
    warnings.warn(f"GT classes absent from model.classes_: {_unknown}")


def _proba_frame(X):
    p = model.predict_proba(X)
    cols = {f"proba_{c}": p[:, i] for i, c in enumerate(MODEL_CLASSES)}
    return pd.DataFrame(cols), p


# --- Item level ---
_pred = [str(v) for v in model.predict(X_ITEM)]
_pf, _p = _proba_frame(X_ITEM)
PRED = ITEMS.copy()
PRED["pred_class"] = _pred
PRED["max_proba"] = _p.max(axis=1)
PRED["correct"] = [
    (pc == tc) if isinstance(tc, str) else np.nan
    for pc, tc in zip(PRED["pred_class"], PRED["true_class"])
]
PRED = pd.concat([PRED.reset_index(drop=True), _pf.reset_index(drop=True)], axis=1)
PRED["classifier_name"] = CLASSIFIER_NAME
PRED["model_path"] = str(MODEL_PATH)
PRED["run_timestamp"] = RUN_TS

for _t in ["field", "pristine", "ood"]:
    _sub = PRED[PRED["tier"] == _t]
    if _sub.empty:
        continue
    _f = RESULTS_DIR / f"{PROJECT_NAME}_predictions_{_t}.csv"
    _sub.to_csv(_f, index=False)
    print(f"saved: {_f.name}  ({len(_sub)} items)")

# --- Pseudo-replicate stability ---
_sp = [str(v) for v in model.predict(X_SPEC)]
_spf, _spp = _proba_frame(X_SPEC)
STAB = SPEC_META.copy()
STAB["pred_class"] = _sp
STAB["max_proba"] = _spp.max(axis=1)
STAB = STAB.sort_values(["tier", "item_key", "id"]).reset_index(drop=True)
STAB["pseudo_replicate_idx"] = STAB.groupby(["tier", "item_key"]).cumcount() + 1
_maj = STAB.groupby(["tier", "item_key"])["pred_class"].agg(
    lambda s: s.value_counts().idxmax()).rename("majority_pred")
STAB = STAB.merge(_maj, on=["tier", "item_key"], how="left")
STAB["agrees_with_majority"] = STAB["pred_class"] == STAB["majority_pred"]
STAB = pd.concat([STAB.reset_index(drop=True), _spf.reset_index(drop=True)], axis=1)
_stab_csv = RESULTS_DIR / f"{PROJECT_NAME}_pseudoreplicate_stability.csv"
STAB.to_csv(_stab_csv, index=False)
print(f"saved: {_stab_csv.name}  ({len(STAB)} spectra)")


# --- Summary ---
def _ovr_brier(sub, cls):
    """One-vs-rest Brier score for one class.

    Not on the same scale as the multiclass Brier score of the benchmark notebooks."""
    col = f"proba_{cls}"
    if col not in sub.columns:
        return np.nan
    y = (sub["true_class"] == cls).astype(float).to_numpy()
    return float(np.mean((y - sub[col].to_numpy()) ** 2))


_rows = []
for _t, g in PRED.groupby("tier"):
    _key = "true_class" if _t != "ood" else "ftir_polymer"
    for _c, sub in g.groupby(_key):
        n = len(sub)
        n_ok = int(sub["correct"].sum()) if sub["correct"].notna().any() else np.nan
        rate = (n_ok / n) if (n >= MIN_ITEMS_FOR_RATE and np.isfinite(n_ok)) else np.nan
        if _t == "ood":
            note = "out-of-distribution polymer, no correct class exists"
        elif n < MIN_ITEMS_FOR_RATE:
            note = f"n={n}, qualitative only"
        else:
            note = ""
        _rows.append({
            "tier": _t,
            "true_class": _c,
            "n_items": n,
            "n_correct": n_ok,
            "accuracy": rate,
            "per_class_brier_ovr": _ovr_brier(g, _c) if _t != "ood" else np.nan,
            "mean_confidence": float(sub["max_proba"].mean()),
            "min_confidence": float(sub["max_proba"].min()),
            "note": note,
        })
SUMMARY = pd.DataFrame(_rows).sort_values(["tier", "true_class"]).reset_index(drop=True)
_sum_csv = RESULTS_DIR / f"{PROJECT_NAME}_summary_by_class.csv"
SUMMARY.to_csv(_sum_csv, index=False)
print(f"saved: {_sum_csv.name}")
print()
display(SUMMARY)


In [ ]:
# === Cell 11: Plot Per-Item Predictions ===
# Reloads the prediction CSVs from RESULTS_DIR; needs only the setup cells and
# shared plot helpers. One figure per tier.

_TIER_TITLE = {"field": "Field", "pristine": "Pristine", "ood": "Out Of Distribution"}
_tiers = [t for t in ["field", "pristine", "ood"]
          if _latest(f"{PROJECT_NAME}_predictions_{t}_*.csv") is not None]

if not _tiers:
    print("no prediction CSVs found in RESULTS_DIR.")
else:
    for t in _tiers:
        d = pd.read_csv(_latest(f"{PROJECT_NAME}_predictions_{t}_*.csv"))
        _key = "true_class" if t != "ood" else "ftir_polymer"
        d = d.sort_values([_key, "item_key"]).reset_index(drop=True)

        # Figure height grows with the longest item label (rotated ticks).
        _lab_h = max(len(str(k)) for k in d["item_key"]) * 0.055
        fig, ax = plt.subplots(figsize=(_FW * 1.15, _FW * 0.50 + _lab_h))
        _add_bands(ax, len(d))
        for i, row in d.iterrows():
            col = MATERIAL_COLORS.get(str(row["pred_class"]), _OTHER_COLOR)
            # ood rows have correct = NaN and are drawn without the "incorrect" hatch.
            ok = bool(row["correct"]) if pd.notna(row.get("correct")) else True
            ax.bar(i, row["max_proba"], color=col, alpha=0.85,
                   edgecolor="black", linewidth=0.3 if ok else 1.4,
                   hatch="" if ok else "///", zorder=2)
            ax.text(i, row["max_proba"] + 0.02, f"{row['max_proba']:.2f}",
                    ha="center", va="bottom", fontsize=_FONT_SIZE - 3, fontweight="bold")
        ax.set_xticks(range(len(d)))
        # Full item labels, rotated vertically to avoid overlap.
        ax.set_xticklabels([str(k) + "  |  " + str(c)
                            for k, c in zip(d["item_key"], d[_key])],
                           rotation=90, fontsize=_FONT_SIZE - 2)
        ax.set_ylim(0, 1.18)
        ax.set_ylabel("Max Predicted Probability")
        ax.set_title(f"{_TIER_TITLE[t]} Tier Per-Item Predictions")
        ax.grid(axis="y", alpha=0.3, zorder=1)
        handles = [Patch(facecolor=MATERIAL_COLORS[c], edgecolor="black", linewidth=0.3,
                         label=f"Predicted {c}") for c in PRED_CLASS_ORDER]
        handles.append(Patch(facecolor="white", edgecolor="black", linewidth=1.4,
                             hatch="///", label="Incorrect"))
        fig.tight_layout()
        # Figure-level legend below the axes, outside the data area.
        fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.02),
                   ncol=3, framealpha=0.9)
        _out = RESULTS_DIR / f"{PROJECT_NAME}_per_item_predictions_{t}.png"
        fig.savefig(_out, dpi=150, bbox_inches="tight",
                    bbox_extra_artists=fig.legends)
        plt.show()
        print(f"saved: {_out.name}")


In [ ]:
# === Cell 12: Plot Confusion Matrices ===
# Raw counts (not row-normalised, since some classes have a single item) for the field
# and pristine tiers; both axes span every predictable class, including Paper.

_tiers = [t for t in ["field", "pristine"]
          if _latest(f"{PROJECT_NAME}_predictions_{t}_*.csv") is not None]

if not _tiers:
    print("no prediction CSVs found in RESULTS_DIR.")
else:
    # Square matrices over PRED_CLASS_ORDER; the Paper row is always zero (no Paper items).
    row_labels = PRED_CLASS_ORDER
    col_labels = PRED_CLASS_ORDER
    fig, axes = plt.subplots(1, len(_tiers), figsize=(_FW, _FW * 0.52))
    axes = np.atleast_1d(axes)
    for k, (ax, t) in enumerate(zip(axes, _tiers)):
        d = pd.read_csv(_latest(f"{PROJECT_NAME}_predictions_{t}_*.csv"))
        cm = np.zeros((len(row_labels), len(col_labels)), dtype=int)
        for _, r in d.iterrows():
            if r["true_class"] in row_labels and r["pred_class"] in col_labels:
                cm[row_labels.index(r["true_class"]), col_labels.index(r["pred_class"])] += 1
        n_dropped = len(d) - cm.sum()
        im = ax.imshow(cm, cmap="Blues", vmin=0)
        for i in range(len(row_labels)):
            for j in range(len(col_labels)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        fontweight="bold",
                        color="white" if cm[i, j] > cm.max() * 0.6 else "black")
        ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=45, ha="right")
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)
        ax.set_box_aspect(1)
        ax.set_xlabel("Predicted Class")
        if k == 0:
            ax.set_ylabel("True Class")
        ax.set_title(f"$\\bf{{({'ab'[k]})}}$ {_TIER_TITLE[t]} Tier"
                     + (f"\n{n_dropped} item(s) outside the shown classes" if n_dropped else ""))
    fig.tight_layout()
    _out = RESULTS_DIR / f"{PROJECT_NAME}_confusion_matrices.png"
    fig.savefig(_out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {_out.name}")


In [ ]:
# === Cell 13: Plot Spectral Overlay ===
# VLIZ item spectra against KI-Sort training class means. Requires TRAINING_SPECTRA_DIR
# (loaded with _load_training_matrix); skipped with a message otherwise.

_Xtr, _ytr, _skip = _load_training_matrix()
if _Xtr is None:
    print(f"no usable training spectra found in {TRAINING_SPECTRA_DIR}.")
else:
    print(f"training spectra: {_Xtr.shape}, skipped {_skip} unresolved files")
    _proc_f = _latest(f"{PROJECT_NAME}_processed_item_spectra_*.csv")
    d = pd.read_csv(_proc_f)
    _wn = np.array([float(c) for c in d.columns[3:]])
    # Wider than the _FW thesis width for readability.
    fig, axes = plt.subplots(2, 2, figsize=(_FW * 1.6, _FW * 0.85), sharex=True)
    for ax, cls in zip(axes.ravel(), MATERIAL_ORDER):
        m = _ytr == cls
        if m.any():
            ax.plot(_wn, _Xtr[m].mean(axis=0), color="black", lw=1.6,
                    label=f"KI-Sort {cls} Mean", zorder=3)
        sub = d[(d["true_class"] == cls) & (d["tier"] != "ood")]
        for _, r in sub.iterrows():
            ax.plot(_wn, r.iloc[3:].to_numpy(dtype=float),
                    color=MATERIAL_COLORS[cls], lw=0.8, alpha=0.6, zorder=2)
        ax.set_title(f"{cls}, n = {len(sub)} VLIZ Items")
        ax.grid(alpha=0.3)
        ax.invert_xaxis()
        ax.legend(loc="best", framealpha=0.8)
    for ax in axes[-1]:
        ax.set_xlabel("Wavenumber (1/cm)")
    for ax in axes[:, 0]:
        ax.set_ylabel("Signal")
    fig.suptitle("VLIZ Item Spectra Against KI-Sort Training Class Means")
    fig.tight_layout()
    _out = RESULTS_DIR / f"{PROJECT_NAME}_spectral_overlay.png"
    fig.savefig(_out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {_out.name}")


In [ ]:
# === Cell 14: Plot Splice Diagnostic ===
# Diagnostic figures (not sized to the _FW thesis width): a raw spectrum with splice
# points and the jump-to-noise ratio per boundary.

_f = _latest(f"{PROJECT_NAME}_splice_diagnostic_*.csv")
if _f is None:
    print("no splice diagnostic CSV found in RESULTS_DIR.")
else:
    sd = pd.read_csv(_f)

    # (a) Representative raw spectrum with the nominal splice points marked.
    _rep_row = int(SEL.iloc[0]["row_idx"])
    fig_a, ax_a = plt.subplots(figsize=(11, 5))
    ax_a.plot(WAVELENGTH_NM, SPECTRA[_rep_row], color="#4a4a4a", lw=1.0)
    ax_a.axvspan(REF_NM_MIN, REF_NM_MAX, color="#2a9d8f", alpha=0.12, zorder=0)
    for wl in SPLICE_WAVELENGTHS_NM:
        ax_a.axvline(wl, color="#e76f51", ls="--", lw=1.4)
    ax_a.set_xlabel("Wavelength (nm)")
    ax_a.set_ylabel("Reflectance")
    ax_a.set_title("Representative Raw Spectrum")
    ax_a.grid(alpha=0.3)
    ax_a.legend(handles=[Line2D([0], [0], color="#e76f51", ls="--",
                                label="Nominal ASD Splice"),
                         Patch(facecolor="#2a9d8f", alpha=0.12,
                               label="KI-Sort Training Window")],
               loc="upper right", framealpha=0.8)
    fig_a.tight_layout()
    _out_a = RESULTS_DIR / f"{PROJECT_NAME}_splice_diagnostic_raw_spectrum.png"
    fig_a.savefig(_out_a, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {_out_a.name}")

    # (b) Jump-to-noise ratio across all selected spectra.
    _cols = [f"jump_ratio_{int(w)}nm" for w in SPLICE_WAVELENGTHS_NM]
    fig_b, ax_b = plt.subplots(figsize=(9, 6))
    _add_bands(ax_b, len(_cols))
    for i, (c, w) in enumerate(zip(_cols, SPLICE_WAVELENGTHS_NM)):
        v = sd[c].dropna().to_numpy()
        _in = bool(REF_NM_MIN <= w <= REF_NM_MAX)
        ax_b.scatter(np.full(v.size, i) + np.random.default_rng(RANDOM_SEED).normal(0, 0.06, v.size),
                    v, s=16, color="#c44e52" if _in else "#7f7f7f", alpha=0.6, zorder=2)
    ax_b.axhline(SPLICE_JUMP_FACTOR, color="#c44e52", ls="--", lw=1.4)
    _xt = []
    for w in SPLICE_WAVELENGTHS_NM:
        _state = "in window" if REF_NM_MIN <= w <= REF_NM_MAX else "cut away"
        _xt.append(str(int(w)) + " nm" + chr(10) + _state)
    ax_b.set_xticks(range(len(_cols)))
    ax_b.set_xticklabels(_xt, fontsize=_FONT_SIZE + 1)
    ax_b.set_ylabel("Jump Magnitude in Noise Units")
    ax_b.set_title("Splice Step Against Local Noise")
    ax_b.grid(axis="y", alpha=0.3)
    ax_b.legend(handles=[Line2D([0], [0], color="#c44e52", ls="--",
                                label=f"Flag Threshold {SPLICE_JUMP_FACTOR:g}")],
               loc="upper right", framealpha=0.8)
    fig_b.tight_layout()
    _out_b = RESULTS_DIR / f"{PROJECT_NAME}_splice_diagnostic_jump_ratio.png"
    fig_b.savefig(_out_b, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {_out_b.name}")


In [ ]:
# === Cell 15: Plot PCA Embedding Overlay ===
# Projects VLIZ items through the classifier's fitted pipeline steps (scaler + PCA).
# Training spectra are overlaid when TRAINING_SPECTRA_DIR is set.


def _embed(X):
    Z = X
    for name, step in model.steps[:-1]:
        Z = step.transform(Z)
    return np.asarray(Z)


_Z_item = _embed(X_ITEM)
if _Z_item.shape[1] < 2:
    print(f"embedding has {_Z_item.shape[1]} component(s); overlay needs 2.")
else:
    _Xtr, _ytr, _ = _load_training_matrix()
    _Ztr = None
    if _Xtr is not None and _Xtr.shape[1] == X_ITEM.shape[1]:
        _Ztr = _embed(_Xtr)
    elif _Xtr is not None:
        print("training spectra feature length differs from the model axis; "
              "training overlay skipped.")

    fig, ax = plt.subplots(figsize=(8 / 2.54, 8 / 2.54))
    if _Ztr is not None:
        for cls in PRED_CLASS_ORDER:
            m = _ytr == cls
            if m.any():
                ax.scatter(_Ztr[m, 0], _Ztr[m, 1], s=14, alpha=0.30,
                           color=MATERIAL_COLORS[cls], edgecolors="none", zorder=1)
    for cls in MATERIAL_ORDER:
        m = (ITEMS["true_class"] == cls).to_numpy()
        if m.any():
            ax.scatter(_Z_item[m, 0], _Z_item[m, 1], s=55, marker="X",
                       color=MATERIAL_COLORS[cls], edgecolors="black",
                       linewidths=0.6, zorder=3)
    m_ood = (ITEMS["tier"] == "ood").to_numpy()
    if m_ood.any():
        ax.scatter(_Z_item[m_ood, 0], _Z_item[m_ood, 1], s=55, marker="X",
                   color=_OTHER_COLOR, edgecolors="black", linewidths=0.6, zorder=3)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.grid(alpha=0.3)
    ax.set_box_aspect(1)
    handles = [Line2D([0], [0], marker="X", color="none", markeredgecolor="black",
                      markerfacecolor=MATERIAL_COLORS[c], markersize=8,
                      label=c) for c in MATERIAL_ORDER]
    if m_ood.any():
        handles.append(Line2D([0], [0], marker="X", color="none", markeredgecolor="black",
                              markerfacecolor=_OTHER_COLOR, markersize=8,
                              label="OOD"))
    if _Ztr is not None:
        handles.append(Line2D([0], [0], marker="o", color="none",
                              markerfacecolor="#999999", markersize=7,
                              label="Training"))
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.18),
             ncol=3, framealpha=0.85, fontsize=_FONT_SIZE - 3)
    fig.tight_layout()
    _out = RESULTS_DIR / f"{PROJECT_NAME}_pca_embedding_overlay.png"
    fig.savefig(_out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {_out.name}")
